## Structured Streaming

What is a stream?

A stream is an unbounded dataset, which has no theoretical beginning or ending.  This is unlike a batch which has a known size and processing time.  Streams can be messages or files processed in real-time as they arrive.

Structured Streaming

- is built on dataframe/dataset APIs
- contains event-time processing
- simplified API with SQL-like operations
- handles late and out of order data
- handled in small 'microbatched' chunks

Key Features of Structured Streaming
- Event-time processing
- Watermark support for late data
- End-to-end exactly once guarantees

In [0]:
# Watermark support in Spark Structured Streaming allows handling late-arriving data.
# It defines a threshold of how late data can be and still be processed.
# Data older than the watermark is considered too late and is dropped.

# Example: Setting watermark on a streaming DataFrame
streaming_df = spark.readStream.format("delta").load("/path/to/stream")
watermarked_df = streaming_df.withWatermark("event_time", "10 minutes")

Autoloader (cloud_files) is a Databricks source for high-performance cloud storage ingestion with auto schema handling

Read Stream Example
DataStreamReader creates streaming DataFrames

```
df = spark.readStream \
    .format("kafka") \
    .option("kafka.boostrap.servers", "host:port") \
    .option("subscribe", "topic1") \
    .load()
```

After the stream is read into a dataframe, one can perform transformations as they normally would

Write stream example with DataStreamWriter

```
query = df.writeStream \
  .format("kafka") \
  .outputMode("append") \
  # .. more options ...
.start()
```

Triggers

- Default Trigger (as soon as possible).  This processes new data as soon as the previous micro-batch completes:  ``` df.writeStream.Start()```

- Fixed Interval Trigger.   Process data at specified time intervals, which is useful for controling resource usages/costs ``` df.writeStream.trigger(processingTime = '2 minuties').start() ```

- Available Now Trigger.  Process available data and then stops, won't wait for more new data to arrive ``` df.writeStream.trigger(availableNow=True).start() ```

Output Modes
- append (default): only adds new records to the sink
- update: modifies existing records and adds new ones. Only outputs records which changed since last trigger
- complete: writes entire result table to sink each time

Streaming Stateful vs Stateless

**Stateless**
- Process each record independently
- No memory of previous records
- Examples:  _select_, _filter_

**Stateful**
- Maintains information across batches
- Require a checkpoint location
- Examples: _groupBy_, _join_, _dropDuplicates_
- Window operations are stateful

Checkpoints

- Maintain state across batches
- Recover state in-case of failures
- Handle replay of data w/o duplicating results

_RocksDB_ is the backend state managing DB
A checkpoint directory is used for maintaining the state metadata

Streaming Joins

- All join types are supported except full and cross
- Streaming dataframes may be joined to other streaming dataframes or to static dataframes
- joins require maintaining state (increases memory usage)

Streaming Aggregations

Streaming datasets are unbound so aggregations are done in _windows_
- Tumbling window:  Fixed non-overlapping intervales (example: count events every 5 mins)
- Sliding windows:  Overlapping windows where an event can belong to multiple windows
- Session windows:  Dynamically sized windows based on user activity, (gaps or session timeouts)